# 16 Standard Curve: Bradford-style Protein Assay - Python

## Biochemistry question

How can a standard curve be used to estimate unknown protein concentrations from absorbance values in a synthetic Bradford-style assay?

This notebook uses synthetic data for learning. The unknown concentrations are practice estimates, not real protein measurements.


In [1]:
import plotly.io as pio
pio.renderers.default = "iframe"


## 1. Setup

We use a synthetic standard curve dataset with known protein standards and unknown samples. The helper functions are in `src/standard_curve.py`.


In [2]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.standard_curve import (
    estimate_unknown_concentrations,
    fit_linear_standard_curve,
    summarize_standard_curve,
    summarize_unknown_estimates,
)


## 2. Dataset Preview

Standards have known concentrations. Unknown samples have blank concentration values and measured absorbance values.


In [3]:
data_path = PROJECT_ROOT / "data" / "standard_curves" / "bradford_standard_curve.csv"
df = pd.read_csv(data_path)
df.head()


,sample_id,sample_type,known_concentration_mg_ml,replicate,absorbance_595
0,STD_000_R1,standard,0.000,1,0.048
1,STD_000_R2,standard,0.000,2,0.052
2,STD_000_R3,standard,0.000,3,0.050
3,STD_0125_R1,standard,0.125,1,0.137
4,STD_0125_R2,standard,0.125,2,0.142


In [4]:
df.groupby("sample_type").size().reset_index(name="row_count")


,sample_type,row_count
0,standard,24
1,unknown,9


## 3. Summarize the Standards

For each known concentration, calculate the mean absorbance and replicate variability.


In [5]:
standard_summary = summarize_standard_curve(df)
standard_summary


,known_concentration_mg_ml,mean_absorbance,sd_absorbance,n,sem_absorbance
0,0.000,0.050000,0.002000,3,0.001155
1,0.125,0.139333,0.002517,3,0.001453
2,0.250,0.231000,0.004000,3,0.002309
3,0.500,0.411667,0.004041,3,0.002333
4,0.750,0.588000,0.004000,3,0.002309
5,1.000,0.764333,0.006506,3,0.003756
6,1.250,0.938333,0.006506,3,0.003756
7,1.500,1.110333,0.008021,3,0.004631


## 4. Fit a Linear Standard Curve

A simple linear model is useful only across the standard range where the assay behaves approximately linearly.


In [6]:
fit = fit_linear_standard_curve(df)
{key: round(value, 4) for key, value in fit.items()}


{'slope': 0.7079,
 'intercept': 0.0535,
 'r_squared': 0.9998,
 'min_standard': 0.0,
 'max_standard': 1.5}

In [7]:
fig = px.scatter(
    standard_summary,
    x="known_concentration_mg_ml",
    y="mean_absorbance",
    error_y="sem_absorbance",
    title="Synthetic Bradford-style Standard Curve",
    labels={
        "known_concentration_mg_ml": "Known concentration (mg/mL)",
        "mean_absorbance": "Mean absorbance at 595 nm",
    },
)
fig.add_scatter(
    x=standard_summary["known_concentration_mg_ml"],
    y=fit["slope"] * standard_summary["known_concentration_mg_ml"] + fit["intercept"],
    mode="lines",
    name="Linear fit",
)
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## 5. Estimate Unknown Concentrations

Unknown sample absorbance values are converted back to concentration estimates using the fitted line.


In [8]:
unknown_estimates = estimate_unknown_concentrations(df, fit)
unknown_estimates[["sample_id", "replicate", "absorbance_595", "estimated_concentration_mg_ml", "outside_standard_range"]]


,sample_id,replicate,absorbance_595,estimated_concentration_mg_ml,outside_standard_range
24,UNK_A_R1,1,0.365,0.440015,False
25,UNK_A_R2,2,0.372,0.449904,False
26,UNK_A_R3,3,0.369,0.445666,False
27,UNK_B_R1,1,0.688,0.896319,False
28,UNK_B_R2,2,0.701,0.914684,False
29,UNK_B_R3,3,0.694,0.904795,False
30,UNK_C_R1,1,1.206,1.628100,True
31,UNK_C_R2,2,1.222,1.650703,True
32,UNK_C_R3,3,1.214,1.639401,True


In [9]:
unknown_summary = summarize_unknown_estimates(unknown_estimates)
unknown_summary.round(3)


,unknown_id,mean_estimated_concentration,sd_estimated_concentration,n,any_outside_standard_range,sem_estimated_concentration
0,UNK_A,0.445,0.005,3,False,0.003
1,UNK_B,0.905,0.009,3,False,0.005
2,UNK_C,1.639,0.011,3,True,0.007


## 6. Check for Extrapolation

Estimates outside the standard range should be treated carefully. In a real lab setting, those samples would often be diluted or re-run within the linear range.


In [10]:
unknown_summary[["unknown_id", "mean_estimated_concentration", "any_outside_standard_range"]]


,unknown_id,mean_estimated_concentration,any_outside_standard_range
0,UNK_A,0.445195,False
1,UNK_B,0.905266,False
2,UNK_C,1.639401,True


## Industry / Lab Reality

In real protein assays, unknowns outside the standard curve range are often diluted and re-run. Teams also check blanks, calibration fit, replicate consistency, dilution factors, and protocol-specific acceptance criteria.

## Career / Job Skill Connection

This notebook practices wet-lab and QC skills used in protein biochemistry, analytical labs, and assay support roles: calibration curves, unknown estimation, interpolation, extrapolation caution, and clear reporting.


## Academic Advancement Practice

Use this notebook to show calibration and measurement reasoning. Prepare one standard-curve figure, one estimated unknown table, one interpolation/extrapolation caution, and one practical lab QC check.

Good academic framing:

- Research assistant angle: explain how a standard curve turns instrument signal into an estimated concentration.
- Lab-program angle: emphasize why unknowns outside the standard range need caution.
- Interview prompt: if an unknown sample is beyond the calibration range, what would you do before reporting a value?

Optional deliverable: complete `writing_templates/figure_caption_template.md` and `writing_templates/limitations_template.md`.


## What You Should Notice

- Absorbance increases as known protein concentration increases.
- The fitted line gives a simple calibration relationship for this synthetic assay.
- Unknown samples can be estimated from the line, but estimates outside the standard range need review.
- A high `r_squared` does not remove the need to check linear range and assay quality.

## Interpretation Practice

- Which unknown sample has the highest estimated concentration?
- Which unknown sample, if any, falls outside the standard range?
- Why is extrapolation risky in a standard curve assay?
- What would you do in a real lab if an unknown sample was above the highest standard?

## Common Mistake

- Do not assume every absorbance value can be converted safely. Standard curves are most useful inside the tested standard range.

## Limitations

- This is synthetic learning data, not a real Bradford assay.
- The notebook uses a simple linear fit and does not model all assay chemistry or instrument behavior.
- Real unknowns may need dilution, re-measurement, blank correction, and protocol-specific review.
- These estimates should not be used for clinical, diagnostic, regulatory, or proprietary conclusions.
